# Incentive exploitation: cohort validation and cost of delayed detection

**Decision this supports:** after how many cycles should accounts that time their activity to a predictable reward schedule be detected and acted on, and what does waiting cost?

All data in this notebook is synthetic, generated with a fixed seed. No figure describes a real programme.

**The programme.** A marketplace runs a recurring cashback pool. Every qualifying spend earns a base cashback and adds to a shared pool. The pool pays out at the end of some step of its cycle, shared pro rata among the spend of that step. The chance of paying out rises predictably the longer the cycle has run, and it always pays by the last step, so concentrating spend late in the cycle earns more than spreading it evenly.

In [ ]:
import pandas as pd

from fraud_analytics import charts as ch
from fraud_analytics.study import run_study

ch.apply_style()
r = run_study()
s = r.summary
prog = s["programme"]
ch.kpis([
    ("Entries", f"{prog['entries']:,}", "one year, 4,000 accounts"),
    ("Complete cycles", f"{prog['complete_cycles']}", f"mean {prog['mean_cycle_steps']:.1f} steps of 4 hours"),
    ("Theoretical return", f"{prog['theoretical_return']['all']:.3f}", "for evenly spread spend"),
])

The **theoretical return** is what an account earns if its spend is spread evenly over time: the base cashback plus the time-average of (pool paid / spend in the paying step). It comes from the programme trajectory alone (no labels, no account outcomes), so the cohorts cannot contaminate it.

Complete cycles are split in two: **discovery** (even cycles) builds the cohorts, **holdout** (odd cycles) only measures them.

## 1. A list built from outcomes validates itself

A common way to assemble a "known exploiters" list is to review the accounts that did well. Here the list is every late-leaning account in the top 5% of discovery return ratios. Measuring its return on the same cycles that put it on the list is circular.

In [ ]:
c, b = s["circularity"], s["behaviour"]
theo = prog["theoretical_return"]["holdout"]
rows = {
    "Outcome-built list": (c["listed_in_sample"]["return"], c["listed_holdout"]["return"]),
    "Behaviour-only cohort": (b["in_sample"]["return"], b["holdout"]["return"]),
}
ch.circularity_dumbbell(rows, theo)
ch.table(pd.DataFrame([
    {"Cohort": "Outcome-built list", "Accounts": c["listed_accounts"],
     "Discovery": ch.fmt_ci(rows["Outcome-built list"][0]), "Holdout": ch.fmt_ci(rows["Outcome-built list"][1])},
    {"Cohort": "Behaviour-only cohort", "Accounts": b["flagged"],
     "Discovery": ch.fmt_ci(rows["Behaviour-only cohort"][0]), "Holdout": ch.fmt_ci(rows["Behaviour-only cohort"][1])},
]), caption=f"Return ratio, 95% BCa interval. Theoretical return on holdout: {theo:.3f}")
ch.note(
    f"The outcome-built list appears to earn **{c['listed_in_sample']['return']['value']:.2f}** per credit on the cycles "
    f"that selected it; on holdout it earns **{c['listed_holdout']['return']['value']:.2f}**, and only "
    f"**{c['listed_share_exploiting']:.0%}** of it are planted exploiters. Most of its apparent premium came from "
    "being selected on it. The behaviour-only cohort holds its level on holdout, because it was never selected on outcomes.")

## 2. Behaviour-only cohort

Exploiters are defined by what they do, never by what they won:

- **Timing concentration:** the share of an account's entries in the late part of the cycle (step at least 60% of the maximum cycle), tested with a one-sided exact binomial test against the share of *time* the programme spends in that part, controlled at a 1% false discovery rate.
- **Exposure breadth:** late entries in at least 3 distinct cycles, so one lucky week cannot qualify an account.

Accounts need at least 8 entries in discovery to be assessed.

In [ ]:
seg = b["by_segment"]
ch.kpis([
    ("Flagged", f"{b['flagged']}", f"of {b['assessed']:,} accounts assessed"),
    ("Precision", f"{b['precision']['value']:.0%}", f"95% CI {b['precision']['ci_low']:.0%} – {b['precision']['ci_high']:.0%}"),
    ("Recall", f"{b['recall']['value']:.0%}", f"of all 120 planted, 95% CI {b['recall']['ci_low']:.0%} – {b['recall']['ci_high']:.0%}"),
])
ch.table(pd.DataFrame([{"Planted segment": k, "Flagged accounts": v} for k, v in seg.items()]),
         caption="Who the flag picks up (planted ground truth)")

## 3. Benchmark: theoretical vs observed return

Observed cohort return against the theoretical return, with a z-test on the pooled (delta-method) standard error that treats accounts, not entries, as the independent units. The **unknown-to-list** cohort is the behaviour-flagged accounts that the outcome-built list missed; its interval is the BCa bootstrap over accounts, on holdout.

In [ ]:
def bench_row(name, cohort):
    bm = cohort["benchmark"]
    return {"Cohort": name, "Accounts": f"{cohort['accounts']:,}", "Observed return (95% BCa)": ch.fmt_ci(cohort["return"]),
            "Theoretical": f"{bm['theoretical']:.3f}", "z": f"{bm['z']:.1f}", "One-sided p": f"{bm['p_value']:.1g}"}

t = s["truth"]
ch.table(pd.DataFrame([
    bench_row("Behaviour-only cohort, holdout", b["holdout"]),
    bench_row("Unknown-to-list cohort, holdout", b["unknown_to_list_holdout"]),
    bench_row("Planted exploiting, all cycles", t["exploiting"]),
    bench_row("Planted heavy legitimate, all cycles", t["heavy"]),
    bench_row("Planted regular, all cycles", t["regular"]),
]), caption="Observed vs theoretical return")

## 4. Monte Carlo time-to-edge

The planted strategy (spend in every step from the middle of the exploiters' active range onwards) is simulated over 4,000 paths of 800 cycles, against the same spend spread evenly. Each cell below is the first cycle from which the probability of a cumulative return of at least +1/+2/+3% stays at or above 50/70/90% **for every later cycle** (sustained crossing).

In [ ]:
mc = s["monte_carlo"]; late = mc["late_strategy"]
tte = late["time_to_edge_cycles"]
ch.edge_curves(r.edge_curves["late_strategy"], tte, r.cfg.mc_cycles)
cell = lambda v: f"{v:,}" if v else f"> {r.cfg.mc_cycles:,}"
ch.table(pd.DataFrame([{"Confidence": conf, **{lvl: cell(tte[lvl][conf]) for lvl in ("+1%", "+2%", "+3%")}}
                       for conf in ("50%", "70%", "90%")]),
         caption="Cycles until the cumulative return stays above the level, by confidence")
ch.note(
    f"The late strategy's long-run return is **{late['expected_return']:.3f}** against **{mc['timing_independent']['expected_return']:.3f}** "
    f"for the same spend spread evenly, which never reaches break-even. A coin-flip chance of being ahead comes within a handful of cycles, "
    f"but being ahead with 90% confidence takes **{tte['+1%']['90%']}** cycles for +1% and **{tte['+3%']['90%']}** for +3%. "
    f"Of the paths, {late['sustained_share_+1%_from_90%_point']:.0%} stay above +1% for every cycle after the 90% point.")

## 5. Cost to the business

Extraction is the reward an account received above what the theoretical return would have paid for the same spend. It is measured on the behaviour-only cohort on **holdout** cycles, so the cohort definition never saw the outcomes it is costed on.

In [ ]:
co = s["cost"]
e, per_cycle, year = co["extraction_per_account_cycle"], co["extraction_cohort_total_per_cycle"], co["extraction_cohort_total_period"]
ch.kpis([
    ("Extraction per exploiter", f"{e['value']:.2f}", f"credits per cycle, 95% CI {e['ci_low']:.2f} – {e['ci_high']:.2f}"),
    ("Cohort, per cycle", ch.fmt_credits(per_cycle["value"]), f"credits, 95% CI {per_cycle['ci_low']:,.0f} – {per_cycle['ci_high']:,.0f}"),
    ("Cohort, full year", ch.fmt_credits(year["value"]), f"credits, 95% CI {year['ci_low']:,.0f} – {year['ci_high']:,.0f}"),
])
ch.note(f"The Monte Carlo predicts **{co['mc_predicted_extraction_per_cycle']:.2f}** credits per exploiter per cycle, below the "
        f"observed {e['value']:.2f}; see the limitations. The cost figures use the observed value.")

### Cost of delayed detection

Loss the cohort accrues before it is detected, if detection happens after N cycles.

In [ ]:
ch.delayed_loss(r.delayed, co["recommended_detection_cycles"])

### Detection-threshold trade-off

A simple operational rule (at least 3 late entries, a late majority, late entries in at least 2 cycles) is applied after N cycles, averaged over windows starting throughout the year. Detecting earlier catches fewer exploiters and flags more legitimate accounts whose few entries happen to fall late. Detecting later lets the exploiters that are caught extract for longer.

Expected cost(N) = extraction x N x exploiters caught + extraction x 40 cycles x exploiters missed + 60 credits x legitimate accounts flagged.

In [ ]:
tr = r.tradeoff
ch.tradeoff(tr)
show = tr[tr["detect_after_cycles"].isin([2, 5, 10, co["recommended_detection_cycles"], 20, 30])]
ch.table(pd.DataFrame({
    "Detect after N cycles": show["detect_after_cycles"].astype(int).astype(str),
    "Exploiters caught (of 120)": show["true_positives"].map("{:.0f}".format),
    "Legitimate accounts flagged": show["false_positives"].map("{:.1f}".format),
    "Expected cost": show["expected_cost"].map("{:,.0f}".format),
}), caption="Selected thresholds (credits; averages over rolling windows)")

### Pool health: what each group puts in and takes out

Every qualifying spend adds 2% to the pool, and the programme adds a 900-credit seed every cycle, so every group gets back more than it puts in. What matters is the gap between groups. The comparison below shows, per account per cycle, how much each group contributes to the pool and how much it receives from it. The second table repeats the split by planted segment, and the last line shows what regular accounts would earn if the exploiters were removed (the same programme re-simulated without them).

In [ ]:
flows = pd.DataFrame(co["pool_flows"])
ch.pool_flows(flows)

def flow_table(df, caption):
    ch.table(pd.DataFrame({
        "Group": df["group"], "Accounts": df["accounts"].map("{:,}".format),
        "Contributed / account / cycle": df["contributed_per_account_cycle"].map("{:.2f}".format),
        "Received / account / cycle": df["received_per_account_cycle"].map("{:.2f}".format),
        "Received per credit contributed": df["received_per_credit_contributed"].map("{:.2f}x".format),
        "Share of contributions": df["share_of_contributions"].map("{:.1%}".format),
        "Share of payouts": df["share_of_awards"].map("{:.1%}".format),
    }), caption=caption)

flow_table(flows, "Pool flows by flag (all complete cycles)")
flow_table(pd.DataFrame(co["pool_flows_truth"]), "Pool flows by planted segment (ground truth)")
w, wo = co["regular_return_with_exploiters"], co["regular_return_without_exploiters"]
ch.note(f"Regular accounts earn **{w['value']:.3f}** per credit with exploiters present and **{wo['value']:.3f}** without them "
        f"(95% CIs {w['ci_low']:.3f} – {w['ci_high']:.3f} and {wo['ci_low']:.3f} – {wo['ci_high']:.3f}).")

## 6. Limitations

- **Synthetic throughout.** Segment behaviour, rates and distributions are invented (see `DECISIONS.md`); real exploiters adapt to detection.
- **The Monte Carlo does not reproduce the observed cohort.** It models one clean strategy against an average population, so it predicts a higher return ratio than the planted cohort achieves (the cohort also spends outside its window and crowds its own window). It also predicts lower extraction per cycle, because real activity is burstier than a fixed spend per step. That gap is behaviour the engine does not capture, so the cost figures use the observed holdout extraction, not the simulation.
- **Cohort-level only.** Per-account returns are too unstable for low-volume accounts to support individual verdicts. The flag is a review queue, not a finding.
- **Recall is limited by volume.** Exploiters with fewer than 8 entries in discovery are never assessed.
- **One seed.** Intervals capture sampling noise within this run, not variation across seeds.
- **Costs are illustrative.** The false-positive cost and the 40-cycle horizon are assumptions; the recommended N moves with them.

## Decision recommendation

In [ ]:
best = tr.loc[tr["expected_cost"].idxmin()]
ch.kpis([
    ("Recommended threshold", f"N = {co['recommended_detection_cycles']}", "cycles of observation"),
    ("Expected cost avoided", ch.fmt_credits(co["expected_cost_avoided"]), "credits vs never acting"),
    ("Cost of one cycle's delay", ch.fmt_credits(per_cycle["value"]), "credits for the cohort"),
])
ch.note(
    f"Detect and act after **{co['recommended_detection_cycles']} cycles** of observation. On this synthetic programme that "
    f"minimises expected cost at about **{co['expected_cost_at_recommended']:,.0f} credits**, against "
    f"**{co['expected_cost_never_detect']:,.0f}** if exploitation is never acted on: an expected "
    f"**{co['expected_cost_avoided']:,.0f} credits avoided**, while flagging about {best['false_positives']:.0f} legitimate "
    f"accounts per review window. Each cycle of delay costs roughly **{per_cycle['value']:,.0f} credits** for the cohort as a whole "
    f"(95% CI {per_cycle['ci_low']:,.0f} – {per_cycle['ci_high']:,.0f}). The cost curve is flat near the minimum, so the exact "
    "choice can follow the review team's capacity. Making the payout schedule less predictable would remove the edge for "
    "everyone and is worth costing alongside detection.")